In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import pickle
import os

PATH = "C:/Users/shata/OneDrive/Desktop/Buildinga job market intelligence platform/"


In [2]:
# ══════════════════════════════════════════════════════════
# LOAD cleaned data
# ══════════════════════════════════════════════════════════

df2 = pd.read_csv(PATH + "data/processed/salaries_clean.csv")

print("Loaded ds_salaries_clean:", df2.shape)
print("Columns:", df2.columns.tolist())


Loaded ds_salaries_clean: (607, 13)
Columns: ['job_title', 'experience_level', 'employment_type', 'salary_avg', 'company_size', 'employee_residence', 'company_location', 'remote_ratio', 'work_mode', 'salary_min', 'salary_max', 'source', 'currency']


In [3]:
# ══════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# Select features that actually predict salary
# ══════════════════════════════════════════════════════════

features = ['job_title', 'experience_level', 'employment_type',
            'company_size', 'company_location', 'remote_ratio']

target = 'salary_avg'

# Drop rows where any feature or target is missing
model_df = df2[features + [target]].dropna()
print(f"\nRows available for training: {len(model_df)}")



Rows available for training: 607


In [4]:
# ══════════════════════════════════════════════════════════
# ENCODE categorical columns → numbers
# ML models only understand numbers, not text
# ══════════════════════════════════════════════════════════

encoders = {}   # save each encoder so we can reuse in dashboard

for col in ['job_title', 'experience_level', 'employment_type',
            'company_size', 'company_location']:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    encoders[col] = le
    print(f"  Encoded '{col}' → {len(le.classes_)} unique values")


  Encoded 'job_title' → 50 unique values
  Encoded 'experience_level' → 4 unique values
  Encoded 'employment_type' → 4 unique values
  Encoded 'company_size' → 3 unique values
  Encoded 'company_location' → 50 unique values


In [5]:
# ══════════════════════════════════════════════════════════
# TRAIN / TEST SPLIT
# 80% train, 20% test
# ══════════════════════════════════════════════════════════

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set : {X_train.shape[0]} rows")
print(f"Testing set  : {X_test.shape[0]} rows")


Training set : 485 rows
Testing set  : 122 rows


In [6]:
# ══════════════════════════════════════════════════════════
# TRAIN THE MODEL — Random Forest Regressor
# ══════════════════════════════════════════════════════════

model = RandomForestRegressor(
    n_estimators=100,    # 100 decision trees
    max_depth=10,        # max depth per tree
    random_state=42,
    n_jobs=-1            # use all CPU cores
)

print("\nTraining model...")
model.fit(X_train, y_train)
print("Training complete!")


Training model...
Training complete!


In [7]:

# ══════════════════════════════════════════════════════════
# EVALUATE — how accurate is the model?
# ══════════════════════════════════════════════════════════

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print("\n" + "=" * 40)
print("MODEL PERFORMANCE")
print("=" * 40)
print(f"  MAE (Mean Absolute Error) : ${mae:,.0f}")
print(f"  R² Score                  : {r2:.3f}")
print(f"\n  Meaning: predictions are off by ~${mae:,.0f} on average")
print(f"  R² of {r2:.2f} means model explains {r2*100:.0f}% of salary variance")



MODEL PERFORMANCE
  MAE (Mean Absolute Error) : $29,648
  R² Score                  : 0.498

  Meaning: predictions are off by ~$29,648 on average
  R² of 0.50 means model explains 50% of salary variance


In [8]:
# ══════════════════════════════════════════════════════════
# FEATURE IMPORTANCE — which factors matter most?
# ══════════════════════════════════════════════════════════

importance_df = pd.DataFrame({
    'feature'   : features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFEATURE IMPORTANCE:")
for _, row in importance_df.iterrows():
    bar = "█" * int(row['importance'] * 50)
    print(f"  {row['feature']:<20} {bar} {row['importance']:.3f}")


FEATURE IMPORTANCE:
  company_location     █████████████████ 0.355
  job_title            ████████████████ 0.335
  experience_level     ████████ 0.168
  company_size         ███ 0.076
  remote_ratio         ██ 0.053
  employment_type       0.014


In [9]:

# ══════════════════════════════════════════════════════════
# TEST PREDICTION — try one example
# ══════════════════════════════════════════════════════════

print("\n" + "=" * 40)
print("SAMPLE PREDICTION TEST")
print("=" * 40)

sample = {
    'job_title'        : 'Data Scientist',
    'experience_level' : 'Senior',
    'employment_type'  : 'Full-Time',
    'company_size'     : 'M',
    'company_location' : 'US',
    'remote_ratio'     : 100
}

# Encode the sample using the same encoders
sample_encoded = []
for col in features:
    if col in encoders:
        val = sample[col]
        # Handle unseen labels gracefully
        if val in encoders[col].classes_:
            sample_encoded.append(encoders[col].transform([val])[0])
        else:
            sample_encoded.append(0)
    else:
        sample_encoded.append(sample[col])

sample_df = pd.DataFrame([sample_encoded], columns=features)
predicted_salary = model.predict(sample_df)[0]

print(f"  Input  : {sample}")
print(f"  Predicted salary: ${predicted_salary:,.0f} USD/year")



SAMPLE PREDICTION TEST
  Input  : {'job_title': 'Data Scientist', 'experience_level': 'Senior', 'employment_type': 'Full-Time', 'company_size': 'M', 'company_location': 'US', 'remote_ratio': 100}
  Predicted salary: $163,002 USD/year


In [10]:
# ══════════════════════════════════════════════════════════
# SAVE model and encoders to models/
# ══════════════════════════════════════════════════════════

os.makedirs(PATH + "models", exist_ok=True)

with open(PATH + "models/salary_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open(PATH + "models/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

# Save feature list so dashboard knows exact column order
with open(PATH + "models/features.pkl", "wb") as f:
    pickle.dump(features, f)

print("\n✅ Saved:")
print("   models/salary_model.pkl")
print("   models/encoders.pkl")
print("   models/features.pkl")


✅ Saved:
   models/salary_model.pkl
   models/encoders.pkl
   models/features.pkl
